# Reroll Failures Analysis

In [2]:
import sqlite3
from pathlib import Path


# Kernel cwd varies by how the notebook was opened (repo root vs. notebooks/);
# resolve the repo root by walking up until we find data/v.db, instead of
# assuming Path.cwd() already is it (matches parselmouth_algo.ipynb).
def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data" / "v.db").exists():
            return candidate
    raise FileNotFoundError(f"could not find data/v.db above {start}")


ROOT = _find_repo_root(Path.cwd())
V_DB = ROOT / "data" / "v.db"  # this repo's PyPI corpus; see the Makefile

# Genuinely read-only: `reroll_data.db.connect(read_only=True)` only skips a
# mkdir, it still opens for writing. A crawl/backfill may be running against
# this file, so a diagnostic notebook must not be able to touch it -- same
# `mode=ro` URI connection as `reroll_data.investigate.connect_ro`.
if not V_DB.is_file():
    raise SystemExit(f"corpus not found: {V_DB}")

v = sqlite3.connect(f"file:{V_DB}?mode=ro", uri=True)
v.execute("PRAGMA busy_timeout=60000")

print("repo root:", ROOT)
print("v.db:", V_DB, f"({V_DB.stat().st_size / 1e9:.2f} GB)")

repo root: /Users/anil/code/reroll-data
v.db: /Users/anil/code/reroll-data/data/v.db (75.21 GB)


## reroll failures

`repodata_conversion.reroll_error` is set by `reroll_convert._convert_one` via
`reroll_index_demo.format_error`: every failure is stored as
`f"{category}: {type(exc).__name__}: {exc}"[:1000]`, where `category` is one
of `reroll_index_demo.CATEGORIES` (`scope`, `invalid`, `unconvertable`,
`runtime`, `unavailable`, `unexpected`) -- see `docs/errors_and_logging.md`
in the sibling `reroll` repo and `reroll.errors` for the full exception
hierarchy behind each category. This is one extra field of structure beyond
`conda_pypi_error` (which has no category prefix): the category is reroll's
own deliberate taxonomy (why the wheel failed *in principle*), so
classification below buckets first by category, then by the concrete
exception leaf class (`reroll.errors`' specific `RerollError` subclasses),
then by message shape within the biggest leaves -- since two wheels raising
the same leaf class can still fail for unrelated concrete reasons (e.g.
`InvalidRequirementError` covers many distinct parser rejections, just like
conda-pypi's `InvalidRequirement`).

In [3]:
### 1. Overview

# Unlike conda-pypi (filtered to `conda_pypi_compatible = 1` wheels by a
# filename pre-check -- see repodata_convert.py:47), reroll gets no filename
# pre-filter (see db.py's `repodata_conversion_reroll_todo` index comment) --
# it is attempted against every row in the table.
n_total = v.execute("SELECT count(*) FROM repodata_conversion").fetchone()[0]
n_attempted = v.execute(
    "SELECT count(*) FROM repodata_conversion "
    "WHERE reroll_data IS NOT NULL OR reroll_error IS NOT NULL"
).fetchone()[0]
n_ok = v.execute(
    "SELECT count(*) FROM repodata_conversion WHERE reroll_data IS NOT NULL"
).fetchone()[0]
n_error = v.execute(
    "SELECT count(*) FROM repodata_conversion WHERE reroll_error IS NOT NULL"
).fetchone()[0]

print(f"total wheel rows:               {n_total:>12,}")
print(
    f"attempted so far:               {n_attempted:>12,}  ({n_attempted / n_total:.1%} of total)"
)
print(
    f"  succeeded:                    {n_ok:>12,}  ({n_ok / n_attempted:.1%} of attempted)"
)
print(
    f"  errored:                      {n_error:>12,}  ({n_error / n_attempted:.1%} of attempted)"
)

total wheel rows:                 12,121,854
attempted so far:                 12,121,854  (100.0% of total)
  succeeded:                       4,472,666  (36.9% of attempted)
  errored:                         7,649,188  (63.1% of attempted)


In [4]:
### 2. Category x exc_type counts

# 7.6M error rows is too much to pull into Python row-by-row (conda-pypi's
# equivalent notebook loaded all 10,400 rows into a DataFrame; that does not
# scale here). Instead, register a small regex UDF and let SQLite do the
# GROUP BY -- only the resulting handful of (category, exc_type) combos
# cross back into Python, this is still one full scan of `reroll_error`
# either way.
import re

import polars as pl

# format_error(): f"{category}: {type(exc).__name__}: {exc}"[:max_len] --
# category is always a lowercase word (see reroll_index_demo.CATEGORIES),
# exc_type an identifier, same shape as conda-pypi's split but with the
# extra category prefix.
_ERR_RE = re.compile(r"^([a-z_]+): ([A-Za-z_][A-Za-z0-9_.]*): (.*)$", re.DOTALL)


def split_error(e: str) -> tuple[str, str, str]:
    m = _ERR_RE.match(e)
    return m.groups() if m else ("UNPARSEABLE", "UNPARSEABLE", e)


def _category(e: str) -> str:
    return split_error(e)[0]


def _exc_type(e: str) -> str:
    return split_error(e)[1]


v.create_function("err_category", 1, _category, deterministic=True)
v.create_function("err_exc_type", 1, _exc_type, deterministic=True)

rows = v.execute(
    "SELECT err_category(reroll_error) AS category, "
    "       err_exc_type(reroll_error) AS exc_type, "
    "       count(*) AS count "
    "FROM repodata_conversion "
    "WHERE reroll_error IS NOT NULL "
    "GROUP BY category, exc_type "
    "ORDER BY count DESC"
).fetchall()

cat_counts = pl.DataFrame(
    rows, schema=["category", "exc_type", "count"], orient="row"
).with_columns((pl.col("count") / n_error).alias("share"))

print(
    f"{len(cat_counts)} distinct (category, exc_type) combos across {n_error:,} errors"
)
with pl.Config(tbl_rows=40):
    display(cat_counts)

22 distinct (category, exc_type) combos across 7,649,188 errors


category,exc_type,count,share
str,str,i64,f64
"""unconvertable""","""UnresolvedCondaNameError""",4519913,0.590901
"""scope""","""UnsupportedPrereleaseError""",1463549,0.191334
"""scope""","""UnsupportedPlatformError""",1023613,0.13382
"""invalid""","""InvalidInterpreterTagError""",244466,0.03196
"""unconvertable""","""UnconvertableRequirementError""",179941,0.023524
"""scope""","""UnsupportedInterpreterError""",142754,0.018663
"""invalid""","""InvalidPythonRequirementRangeE…",33427,0.00437
"""unconvertable""","""PythonRangeMismatchError""",22283,0.002913
"""invalid""","""InvalidRequirementError""",7169,0.000937


In [5]:
for r in cat_counts.iter_rows(named=True):
    print(
        f"{r['category']:<14} {r['exc_type']:<32} {r['count']:>10,}  {r['share']:>7.2%}"
    )

unconvertable  UnresolvedCondaNameError          4,519,913   59.09%
scope          UnsupportedPrereleaseError        1,463,549   19.13%
scope          UnsupportedPlatformError          1,023,613   13.38%
invalid        InvalidInterpreterTagError          244,466    3.20%
unconvertable  UnconvertableRequirementError       179,941    2.35%
scope          UnsupportedInterpreterError         142,754    1.87%
invalid        InvalidPythonRequirementRangeError     33,427    0.44%
unconvertable  PythonRangeMismatchError             22,283    0.29%
invalid        InvalidRequirementError               7,169    0.09%
scope          UnsupportedInterpreterVersionError      4,236    0.06%
unconvertable  UnconvertableMarkerError              3,453    0.05%
invalid        MetadataFilenameMismatchError         1,220    0.02%
unconvertable  InvalidCondaNameError                 1,085    0.01%
invalid        InvalidVersionSpecifierError            992    0.01%
unavailable    MetadataUnavailable          

In [6]:
### 3. Sample messages per (category, exc_type), to design sub-bucket rules

# LIKE 'category: exc_type:%' is a prefix match on the exact `format_error`
# shape, so SQLite can use it directly (no UDF call needed for filtering);
# LIMIT means common buckets short-circuit the scan instead of paying for a
# full one. Ordered by count so the biggest, most-impactful buckets are
# inspected first.
samples: dict[tuple[str, str], list[str]] = {}
with pl.Config(fmt_str_lengths=200):
    for r in cat_counts.iter_rows(named=True):
        cat, exc_type = r["category"], r["exc_type"]
        like = f"{cat}: {exc_type}:%"
        msgs = [
            row[0]
            for row in v.execute(
                "SELECT reroll_error FROM repodata_conversion "
                "WHERE reroll_error LIKE ? LIMIT 6",
                (like,),
            ).fetchall()
        ]
        samples[(cat, exc_type)] = msgs
        print(f"=== {cat}: {exc_type} ({r['count']:,}) ===")
        for m in msgs[:6]:
            print(" ", m[:220].replace("\n", " \\n "))
        print()

=== unconvertable: UnresolvedCondaNameError (4,519,913) ===
  unconvertable: UnresolvedCondaNameError: no mapper resolved a conda name for 'fastapi': candidates=(Candidate(conda_name='fast
api', probability=0.95, source=<CandidateSource.PARSELMOUTH: 'parselmouth'>, mapper='parselmou
  unconvertable: UnresolvedCondaNameError: no mapper resolved a conda name for 'fastapi': candidates=(Candidate(conda_name='fast
api', probability=0.95, source=<CandidateSource.PARSELMOUTH: 'parselmouth'>, mapper='parselmou
  unconvertable: UnresolvedCondaNameError: no mapper resolved a conda name for 'fastapi': candidates=(Candidate(conda_name='fast
api', probability=0.95, source=<CandidateSource.PARSELMOUTH: 'parselmouth'>, mapper='parselmou
  unconvertable: UnresolvedCondaNameError: no mapper resolved a conda name for 'fastapi': candidates=(Candidate(conda_name='fast
api', probability=0.95, source=<CandidateSource.PARSELMOUTH: 'parselmouth'>, mapper='parselmou
  unconvertable: UnresolvedCondaNameError: n

In [7]:
# Closer look at the single biggest bucket (59% of all errors): does
# "no mapper resolved" mean literally no candidate, or a candidate that got
# rejected for some other reason (e.g. below a probability threshold)?
rows = v.execute(
    "SELECT project, filename, reroll_error FROM repodata_conversion "
    "WHERE reroll_error LIKE 'unconvertable: UnresolvedCondaNameError:%' LIMIT 20"
).fetchall()
for project, filename, err in rows:
    print(project, "|", filename)
    print(" ", err)
    print()

01OS | 01os-0.0.1-py3-none-any.whl
  unconvertable: UnresolvedCondaNameError: no mapper resolved a conda name for 'fastapi': candidates=(Candidate(conda_name='fast
api', probability=0.95, source=<CandidateSource.PARSELMOUTH: 'parselmouth'>, mapper='parselmouth_relations'), Candidate(conda_na
me='fastapi-core', probability=0.95, source=<CandidateSource.PARSELMOUTH: 'parselmouth'>, mapper='parselmouth_relations'))

01OS | 01os-0.0.10-py3-none-any.whl
  unconvertable: UnresolvedCondaNameError: no mapper resolved a conda name for 'fastapi': candidates=(Candidate(conda_name='fast
api', probability=0.95, source=<CandidateSource.PARSELMOUTH: 'parselmouth'>, mapper='parselmouth_relations'), Candidate(conda_na
me='fastapi-core', probability=0.95, source=<CandidateSource.PARSELMOUTH: 'parselmouth'>, mapper='parselmouth_relations'))

01OS | 01os-0.0.11-py3-none-any.whl
  unconvertable: UnresolvedCondaNameError: no mapper resolved a conda name for 'fastapi': candidates=(Candidate(conda_name='fast


**Likely bug found**: `fastapi` has a `probability=0.95` candidate from a
*single* mapper (`parselmouth_relations`) yet is still "unresolved"; so does
`pillow` at `probability=0.9413`. Reading `reroll.name_mapping.aggregator_mapper`
(`/Users/anil/code/reroll/src/reroll/name_mapping.py:105-112`):

```python
if len({candidate.mapper for candidate in candidates}) == 1:
    if candidates[0].source is CandidateSource.PARSELMOUTH:
        if len(candidates) == 1:                  # <-- only accepts a LONE candidate
            return candidates[0].conda_name
    else:
        best = max(candidates, key=lambda c: c.probability)
        if best.probability >= 0.9:                # <-- the >=0.9 threshold ...
            return best.conda_name                 #     ... never applies to parselmouth
return candidates
```

The docstring promises "a sole mapper's candidate scoring at least 0.9, **or**
parselmouth's only candidate" -- two independent acceptance rules. The `if
source is PARSELMOUTH: ... else: ...` structure instead makes them mutually
exclusive: the probability-threshold branch only runs for non-parselmouth
sources, and the parselmouth branch only accepts `len(candidates) == 1`. So
any single-mapper parselmouth result with *more than one* candidate (however
confident the best one is) always falls through to `return candidates`, i.e.
`map_name` raises `UnresolvedCondaNameError` -- **even when the top
candidate is a well above the 0.9 bar that a non-parselmouth mapper would be
held to.** Quantify how much of the 4.52M-row bucket this explains below.

In [8]:
### 4a. Sub-bucket UnresolvedCondaNameError: how much is the aggregator_mapper bug?

_CAND_RE = re.compile(
    r"mapper='([^']*)'.*?probability=([0-9.]+).*?mapper='([^']*)'|"
    r"probability=([0-9.]+),\s*source=<CandidateSource\.(\w+):[^>]*>,\s*mapper='([^']*)'"
)
# simpler/robust: pull (probability, source, mapper) triples directly.
_TRIPLE_RE = re.compile(
    r"probability=([0-9.]+),\s*source=<CandidateSource\.(\w+):[^>]*>,\s*mapper='([^']*)'"
)


def classify_unresolved(msg: str) -> str:
    triples = _TRIPLE_RE.findall(msg)
    if not triples:
        return "no_candidates"
    mappers = {m for _, _, m in triples}
    sources = {s for _, s, _ in triples}
    probs = [float(p) for p, _, _ in triples]
    best = max(probs)
    if len(mappers) == 1:
        if sources == {"PARSELMOUTH"} and len(triples) > 1 and best >= 0.9:
            return "aggregator_bug_high_confidence_ignored"
        if best < 0.9:
            return "single_mapper_low_confidence"
        return "single_mapper_other"
    # >=2 distinct mappers: _vote_winner needs 2+ *mappers* to agree on the
    # SAME conda_name; distinct mappers disagreeing on the name is the
    # remaining "real" no-consensus case.
    return "multi_mapper_no_consensus"


v.create_function("classify_unresolved", 1, classify_unresolved, deterministic=True)

rows = v.execute(
    "SELECT classify_unresolved(reroll_error) AS bucket, count(*) AS count "
    "FROM repodata_conversion "
    "WHERE reroll_error LIKE 'unconvertable: UnresolvedCondaNameError:%' "
    "GROUP BY bucket ORDER BY count DESC"
).fetchall()
for bucket, count in rows:
    print(f"{bucket:<38} {count:>10,}  {count / n_error:>7.2%} of all errors")

aggregator_bug_high_confidence_ignored  4,419,408   57.78% of all errors
single_mapper_low_confidence              100,502    1.31% of all errors
multi_mapper_no_consensus                       3    0.00% of all errors


**Confirmed directly against `reroll`'s own source** (not just inferred from
the stored message), run from this repo's `.venv` (`py-reroll` is an
editable install of `/Users/anil/code/reroll`):

```python
from reroll.name_mapping import Candidate, CandidateSource, aggregator_mapper, map_name
from packaging.utils import canonicalize_name

candidates = (
    Candidate(conda_name='fastapi', probability=0.95, source=CandidateSource.PARSELMOUTH, mapper='parselmouth_relations'),
    Candidate(conda_name='fastapi-core', probability=0.95, source=CandidateSource.PARSELMOUTH, mapper='parselmouth_relations'),
)
aggregator_mapper(canonicalize_name('fastapi'), candidates)
# -> returns `candidates` unchanged (should return 'fastapi')
map_name('fastapi', (stub_mapper_returning_those_two_candidates, aggregator_mapper))
# -> raises UnresolvedCondaNameError
```

And end-to-end against a real corpus row --
`reroll_index_demo.wheel_to_records("data/v.db", "01os-0.0.1-py3-none-any.whl", project="01OS")`
-- raises the exact same `UnresolvedCondaNameError` stored in this row of
`repodata_conversion`. **This single bug explains 57.8% of every reroll
failure in this corpus (4,419,408 / 7,649,188 rows)** -- by far the largest
class of failure found in this whole analysis, dwarfing every other bucket
below.

**Suggested fix** (for a follow-up PR against `reroll`, not this analysis):
in `aggregator_mapper`, apply the `probability >= 0.9` check to the
single-mapper case regardless of `source`, e.g.

```python
if len({candidate.mapper for candidate in candidates}) == 1:
    best = max(candidates, key=lambda c: c.probability)
    if candidates[0].source is CandidateSource.PARSELMOUTH and len(candidates) == 1:
        return candidates[0].conda_name
    if best.probability >= 0.9:
        return best.conda_name
```

**Unit test to add** (`tests/test_name_mapping.py` in the `reroll` repo):
assert `aggregator_mapper(name, candidates)` returns `"fastapi"` for the
two-`Candidate` `fastapi`/`fastapi-core` tuple above (both `probability=0.95`,
single mapper) -- currently it returns `candidates` unchanged.

In [9]:
### 4b. Sub-bucket UnconvertableRequirementError (its docstring lists 5 distinct reasons)


def classify_unconvertable_requirement(msg: str) -> str:
    if "it is a pre-release and allow_pre is unset" in msg:
        return "pre_release_dependency"
    if "local version label" in msg:
        return "local_version_label"
    if "direct URL" in msg or "@ http" in msg or " @ " in msg:
        return "direct_url_reference"
    if "extra" in msg and "64 characters" in msg:
        return "extra_name_too_long"
    if "marker" in msg and "extra" in msg:
        return "marker_refers_to_extra"
    if "MatchSpec" in msg or "matchspec" in msg or "validation" in msg:
        return "matchspec_validation_failed"
    return "other"


v.create_function(
    "classify_ucr", 1, classify_unconvertable_requirement, deterministic=True
)

rows = v.execute(
    "SELECT classify_ucr(reroll_error) AS bucket, count(*) AS count "
    "FROM repodata_conversion "
    "WHERE reroll_error LIKE 'unconvertable: UnconvertableRequirementError:%' "
    "GROUP BY bucket ORDER BY count DESC"
).fetchall()
for bucket, count in rows:
    print(f"{bucket:<28} {count:>9,}  {count / n_error:>7.2%} of all errors")

pre_release_dependency         178,286    2.33% of all errors
matchspec_validation_failed        905    0.01% of all errors
local_version_label                656    0.01% of all errors
direct_url_reference                50    0.00% of all errors
extra_name_too_long                 44    0.00% of all errors


### 4c. The next-largest class: `allow_pre` is never passed by this repo's batch job

`scope: UnsupportedPrereleaseError` (1,463,549, 19.1%) + `unconvertable:
UnconvertableRequirementError`'s `pre_release_dependency` sub-bucket
(178,286, 2.3%) = **1,641,835 rows (21.5% of all errors)**, and both stem
from the same root cause: `reroll.wheel_record.get_wheel_records` and
`reroll.pep508_to_matchspec` both default `allow_pre=False`
(`/Users/anil/code/reroll/src/reroll/wheel_record.py:48`), and
`reroll_index_demo._entry_from_db` -- the only place this corpus calls
`get_wheel_records` -- never passes `allow_pre` at all:

```python
records = get_wheel_records(
    metadata, fn, mappers=mappers, sha256=wheel_sha256, size=size, url=url
)
```

Unlike the `aggregator_mapper` bug above, this is **not a reroll bug** --
`docs/errors_and_logging.md`'s scope category is deliberate, and `allow_pre`
existing as an opt-in is the documented way callers get pre-release wheels
converted anyway. It is a gap in *this* repo's batch harness
(`reroll_index_demo`/`reroll_convert`), not `reroll` itself: a caller that
wants pre-release wheels converted should thread `allow_pre=True` through
`_entry_from_db` (and probably make it a CLI flag on `reroll-data reroll
convert`, since always-True would silently accept unstable pins other
callers may not want). Filed here anyway because it is the second-largest
share of every observed failure, and -- unlike the categories below --
fixing it requires no reroll change, just wiring an existing flag through.

### 5. The `unexpected` category: every row here is, by definition, a bug

`reroll_index_demo.categorize_error` falls back to `"unexpected"` for
*anything* that is not a `RerollError` subclass -- i.e. every row in this
category is a case where `reroll` (or a library it calls) raised an
exception it should have caught and wrapped in one of its four documented
categories, but didn't. Small in volume (352 rows total, 0.005% of all
errors) but each is a distinct, fixable gap in `reroll`'s own error
handling -- unlike the categories above, these are not "reroll working as
designed", they're `reroll` leaking. Four leaf types appear; look at each.

In [10]:
for exc_type in [
    "IndexError",
    "ValidationError",
    "UnicodeDecodeError",
    "InvalidRequirement",
]:
    print(f"=== unexpected: {exc_type} ===")
    rows = v.execute(
        "SELECT project, filename, reroll_error FROM repodata_conversion "
        "WHERE reroll_error LIKE ? LIMIT 3",
        (f"unexpected: {exc_type}:%",),
    ).fetchall()
    for project, filename, err in rows:
        print(f"  {project} | {filename}")
        print(f"    {err[:300]}")
    print()

=== unexpected: IndexError ===
  ast-serialize | ast_serialize-0.7.0-cp315-abi3.abi3t-macosx_10_12_x86_64.whl
    unexpected: IndexError: list index out of range
  ast-serialize | ast_serialize-0.7.0-cp315-abi3.abi3t-macosx_11_0_arm64.whl
    unexpected: IndexError: list index out of range
  ast-serialize | ast_serialize-0.7.0-cp315-abi3.abi3t-manylinux_2_17_aarch64.manylinux2014_aarch64.whl
    unexpected: IndexError: list index out of range

=== unexpected: ValidationError ===
  HolyGrail | HolyGrail-0.2.1.Perceval-py2-none-any.whl
    unexpected: ValidationError: 1 validation error for WheelMetadata
version
  Value error, Invalid version: '0.2.1.Perceval' [type=value_error, input_value='0.2.1.Perceval', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
  RHVoice | RHVoice-0.5.1.5-py2-none-win32.whl
    unexpected: ValidationError: 2 validation errors for WheelMetadata
name
  Input should be a valid string [type=string_type, input_value=

In [11]:
### 5a. Root-cause the `unexpected` leaks (each reproduced directly against reroll's source, from this repo's .venv)

# unexpected: IndexError -- reroll/filename/__init__.py:112 `raise errors[-1]`
# assumes `errors` is non-empty whenever `configs` is empty. That invariant
# breaks when `tags` is empty *before* the per-tag loop even runs (so
# `errors` never gets appended to) -- which happens for a compound abi tag
# like `abi3.abi3t` on an interpreter version (`cp315`) past
# `abi3_upper_bound`: `packaging.utils.parse_wheel_filename` splits
# `abi3.abi3t` into two separate tags, and `explode_abi3` filters both out
# entirely, leaving `tags` (and hence `errors`) empty.
from packaging.utils import parse_wheel_filename

name, version, build, tags = parse_wheel_filename(
    "ast_serialize-0.7.0-cp315-abi3.abi3t-macosx_10_12_x86_64.whl"
)
print("parsed tags:", list(tags))
print(
    "(this is the exact IndexError repro -- see reroll/filename/abi3.py's "
    "explode_abi3, which returns frozenset() for all of these)"
)

parsed tags: [<cp315-abi3-macosx_10_12_x86_64 @ 4681399296>, <cp315-abi3t-macosx_10_12_x86_64 @ 4681532224>]
(this is the exact IndexError repro -- see reroll/filename/abi3.py's explode_abi3, which returns frozenset() for all of these)


**Fix**: in `reroll/filename/__init__.py`, `parse_filename` (around line 112),
`if not configs: raise errors[-1]` needs an `errors` empty-case, e.g.
`raise errors[-1] if errors else InvalidFilenameError(f"no supported (tag, arch) for {filename!r}")`.

**Unit test to add** (`tests/test_filename.py` in `reroll`):
`parse_filename("ast_serialize-0.7.0-cp315-abi3.abi3t-macosx_10_12_x86_64.whl", mappers)`
should raise a `RerollError` (e.g. `InvalidFilenameError`/a scope error), not `IndexError`.
Repro project/filename in this corpus: `ast-serialize` /
`ast_serialize-0.7.0-cp315-abi3.abi3t-macosx_10_12_x86_64.whl` (and its sibling
platform wheels -- `SELECT * FROM repodata_conversion WHERE reroll_error LIKE
'unexpected: IndexError:%'` for the rest, all the same shape).

---

**`unexpected: ValidationError`** (42 rows) -- traced (via `.venv/bin/python`
`traceback.print_exc()` against
`reroll_index_demo.wheel_to_records("data/v.db", "HolyGrail-0.2.1.Perceval-py2-none-any.whl", project="HolyGrail")`)
to `reroll/wheel_metadata.py`'s `parse_metadata`:

```python
return WheelMetadata.model_validate({...})   # no try/except
```

`WheelMetadata`'s `version: PyVersion` field validator rejects `'0.2.1.Perceval'`
(not a valid PEP 440 version), and *every* other field-validation failure in
this model (`name`'s `_normalize_dist_name` is the one exception -- it
explicitly catches and re-raises as `InvalidMetadataError`) raises a bare
`pydantic_core.ValidationError` straight out of `model_validate`, uncaught.

**Fix**: wrap the `model_validate(...)` call in `parse_metadata` in a
try/except for `pydantic.ValidationError`, re-raising as
`reroll.errors.InvalidMetadataError` (the same category `_normalize_dist_name`
already uses for a bad `Name`) with the validation error's own message.

**Unit test to add** (`tests/test_wheel_metadata.py`): `parse_metadata(...)`
with a `METADATA` body containing `Version: 0.2.1.Perceval` should raise
`InvalidMetadataError`, not `pydantic.ValidationError`. Repro: `HolyGrail` /
`HolyGrail-0.2.1.Perceval-py2-none-any.whl`.

---

**`unexpected: UnicodeDecodeError`** (31 rows) -- traced to
**this repo** (`reroll_data`), not `reroll` itself:
`reroll_data/conda_pypi_index_demo.py:158`

```python
return zlib.decompress(z_body).decode("utf-8")
```

Some old wheels' `METADATA` (PEP 658 sidecar) is not valid UTF-8 (pre-2010s
PyPI uploads commonly used Latin-1/cp1252 for non-ASCII author names in
`Author`/`Maintainer` headers). `_metadata_body` hardcodes `"utf-8"` with no
fallback or `errors=` policy, so this raises before `reroll` is ever
reached -- meaning it isn't even a `reroll` correctness question, but this
corpus's own harness needs, at minimum, `errors="replace"` (or a
latin-1/cp1252 fallback decode) so these wheels reach `reroll` at all.

**Unit test to add** (in `reroll_data`, `tests/test_conda_pypi_index_demo.py`
or similar): `_metadata_body` on a stored blob with a non-UTF-8 byte should
not raise. Repro: `EmailDepute` / `EmailDepute-1.1.2021-py2-none-any.whl`.

---

**`unexpected: InvalidRequirement`** (26 rows) -- the deepest chain found in
this analysis, root-caused across *two* repos:

1. Raw `METADATA`: `Requires-Dist: ordereddict (==1.1); extra ==
   ':python_version=="2.6"'` -- pre-PEP-508 setuptools convention abusing
   the `extra` marker to smuggle a `python_version` conditional (`bcdoc`
   0.15.0, uploaded 2015).
2. `packaging`'s own `Marker` parser PEP-685-normalizes *any* `extra ==`
   literal via `canonicalize_name` at parse time -- turning
   `:python_version=="2.6"` into `:python-version=="2-6"` -- which is
   `packaging`'s documented (if surprising here) behavior, not a bug: this
   step round-trips fine on its own.
3. `reroll/dependencies/conditional_dependency.py:93`,
   `return str(tighten_ranges(partial_evaluation))`, stringifies the
   *residual* marker node through **`markerpry`** (a separate dependency,
   checked out at `/Users/anil/code/markerpry`).
4. `markerpry/markerpry/node.py:99` (`CompareNode.__str__`):
   `return f'{self.key} {self.comparator} "{self.literal}"'` --
   unconditionally wraps `self.literal` in a **new** pair of double quotes,
   with no escaping. Since `self.literal` here is already
   `:python-version=="2-6"` (containing its own literal `"` characters from
   step 2), the result is the malformed
   `extra == ":python-version=="2-6""` -- 4 unescaped quote characters,
   unparseable as PEP 508.
5. `reroll/dependencies/calculate_dependencies.py:131-133` folds that string
   into `final_entry`, which `reroll/dependencies/pep508_to_matchspec.py:48`
   (`requirement = Requirement(entry)`, no try/except) re-parses --
   raising the bare `packaging.requirements.InvalidRequirement` that leaks
   all the way up as `unexpected`.

**Fix, two independent options** (either alone closes this class):
- `markerpry`'s `CompareNode.__str__`/its RHS-key sibling (`node.py:99,119-120`)
  should escape (or `repr()`-quote) `self.literal` instead of always
  wrapping it in a literal `"..."` -- this is the actual bug, since any
  literal containing a `"` breaks round-tripping, not just this marker-abuse
  case.
- Independently, `reroll`'s own `pep508_to_matchspec.py:48` (and
  `calculate_dependencies.py:126`, `requires_dist.py:26`, `extras.py:30`,
  every other bare `Requirement(entry)` call outside `wheel_metadata.py`)
  should catch `packaging.requirements.InvalidRequirement` and re-raise as
  `reroll.errors.InvalidRequirementError`, the same way `wheel_metadata.py`
  already does via `parse_lenient_requirement` -- so *any* future
  reconstruction bug like this degrades to a correctly-categorized `invalid`
  error instead of leaking as `unexpected`.

**Unit test to add** (`markerpry`, `tests/test_node.py`): round-trip a
`CompareNode(key="extra", comparator="==", literal='foo"bar')` through
`str()` and back through `parse_marker`/`Marker()` and assert it survives,
or at least that `str()` produces a parseable marker. Repro in this corpus:
`bcdoc` / `bcdoc-0.15.0-py2.py3-none-any.whl`.

In [12]:
### 6. Sub-bucket the remaining large *deliberate scope/invalid* buckets, by the concrete tag value

# These are reroll working as designed (docs/errors_and_logging.md's scope
# and invalid categories), not bugs -- but breaking them down by the actual
# offending tag shows *what* is unsupported, concretely.
_TAG_RE = re.compile(r": '([^']*)'\s*$")


def extract_tag(msg: str) -> str:
    m = _TAG_RE.search(msg.split("\n", 1)[0])
    return m.group(1) if m else "?"


v.create_function("extract_tag", 1, extract_tag, deterministic=True)

for cat, exc_type, label in [
    ("scope", "UnsupportedPlatformError", "platform tag"),
    ("invalid", "InvalidInterpreterTagError", "interpreter tag"),
    ("scope", "UnsupportedInterpreterError", "interpreter major"),
    ("scope", "UnsupportedInterpreterVersionError", "cpython tag"),
]:
    rows = v.execute(
        "SELECT extract_tag(reroll_error) AS tag, count(*) AS count "
        "FROM repodata_conversion WHERE reroll_error LIKE ? "
        "GROUP BY tag ORDER BY count DESC LIMIT 12",
        (f"{cat}: {exc_type}:%",),
    ).fetchall()
    print(f"=== {cat}: {exc_type} ({label}) ===")
    for tag, count in rows:
        print(f"  {tag!r:<20} {count:>9,}")
    print()

=== scope: UnsupportedPlatformError (platform tag) ===
  'win32'                201,032
  'musllinux_1_2_x86_64'   136,578
  'musllinux_1_2_aarch64'    93,008
  'manylinux_2_5_i686'    63,818
  'musllinux_1_2_i686'    58,355
  'musllinux_1_1_x86_64'    55,599
  'manylinux_2_17_ppc64le'    53,845
  'manylinux_2_17_i686'    53,704
  'manylinux_2_17_armv7l'    52,472
  'manylinux_2_17_s390x'    49,443
  'musllinux_1_2_armv7l'    36,656
  'musllinux_1_1_i686'    29,409

=== invalid: InvalidInterpreterTagError (interpreter tag) ===
  'pp39'                  62,412
  'pp310'                 61,493
  'pp311'                 43,861
  'pp38'                  42,137
  'pp37'                  28,063
  'pp36'                   3,202
  'pp27'                   1,441
  'cp3'                      227
  'graalpy312'               222
  '3'                        142
  'graalpy311'               137
  'pp35'                     121

=== scope: UnsupportedInterpreterError (interpreter major) ===
  'py2'

In [13]:
### 7. Sub-bucket invalid: InvalidRequirementError by the packaging parser's reason line
# (Same rule set as the conda-pypi notebook's InvalidRequirement bucket --
# reroll's own `parse_lenient_requirement` wraps the identical
# `packaging.requirements.InvalidRequirement` after exhausting its fixups.)

_INVALID_REQ_RULES = [
    (
        "mismatched_parenthesis",
        r"Expected matching RIGHT_PARENTHESIS for LEFT_PARENTHESIS",
    ),
    ("wildcard_suffix_misuse", r"\.\* suffix can only be used"),
    ("expected_package_name", r"Expected package name at the start"),
    ("expected_marker_or_string", r"Expected a marker variable or quoted string"),
    ("expected_end_or_semicolon", r"Expected end or semicolon"),
    ("invalid_specifier", r"Invalid specifier:"),
    ("local_version_label_misuse", r"Local version label can only be used"),
    ("expected_comma_or_end", r"Expected comma or end"),
    ("expected_string", r"Expected string"),
    ("expected_version", r"Expected version"),
    ("expected_end_of_specifier", r"Expected end of dependency specifier"),
]


def classify_invalid_requirement(msg: str) -> str:
    # message shape: "invalid requirement '<entry>': <packaging reason>\n    <entry>\n    ~~^"
    reason = msg.split(": ", 1)[-1] if ": " in msg else msg
    for sub_type, pattern in _INVALID_REQ_RULES:
        if re.search(pattern, reason):
            return sub_type
    return "other"


v.create_function("classify_ir", 1, classify_invalid_requirement, deterministic=True)

rows = v.execute(
    "SELECT classify_ir(reroll_error) AS bucket, count(*) AS count "
    "FROM repodata_conversion WHERE reroll_error LIKE 'invalid: InvalidRequirementError:%' "
    "GROUP BY bucket ORDER BY count DESC"
).fetchall()
for bucket, count in rows:
    print(f"{bucket:<28} {count:>7,}  {count / n_error:>7.3%} of all errors")

mismatched_parenthesis         6,762   0.088% of all errors
local_version_label_misuse       262   0.003% of all errors
wildcard_suffix_misuse           127   0.002% of all errors
expected_end_or_semicolon         11   0.000% of all errors
expected_package_name              5   0.000% of all errors
expected_marker_or_string          2   0.000% of all errors


In [14]:
### 8. Summary: every reroll failure bucket, largest first, with the "is this a bug?" call

summary_rows = [
    # (bucket, category, exc_type, count, verdict)
    (
        "aggregator_mapper_ignores_multi_candidate_parselmouth",
        "unconvertable",
        "UnresolvedCondaNameError",
        4_419_408,
        "BUG (reroll)",
    ),
    (
        "single_mapper_low_confidence",
        "unconvertable",
        "UnresolvedCondaNameError",
        100_502,
        "expected",
    ),
    (
        "wheel_version_prerelease_allow_pre_unset",
        "scope",
        "UnsupportedPrereleaseError",
        1_463_549,
        "config gap (reroll_data)",
    ),
    (
        "unsupported_platform_tag",
        "scope",
        "UnsupportedPlatformError",
        1_023_613,
        "expected (scope)",
    ),
    (
        "invalid_interpreter_tag_pypy_etc",
        "invalid",
        "InvalidInterpreterTagError",
        244_466,
        "expected (invalid)",
    ),
    (
        "dependency_prerelease_allow_pre_unset",
        "unconvertable",
        "UnconvertableRequirementError",
        178_286,
        "config gap (reroll_data)",
    ),
    (
        "unsupported_interpreter_major_py2",
        "scope",
        "UnsupportedInterpreterError",
        142_754,
        "expected (scope)",
    ),
    (
        "non_contiguous_python_range",
        "invalid",
        "InvalidPythonRequirementRangeError",
        33_427,
        "expected (invalid)",
    ),
    (
        "filename_requires_python_mismatch",
        "unconvertable",
        "PythonRangeMismatchError",
        22_283,
        "expected (unconvertable)",
    ),
    (
        "invalid_requirement_mismatched_parenthesis",
        "invalid",
        "InvalidRequirementError",
        6_762,
        "expected (bad upstream metadata)",
    ),
    (
        "unresolved_multi_mapper_no_consensus",
        "unconvertable",
        "UnresolvedCondaNameError",
        3,
        "expected",
    ),
    (
        "unsupported_cpython_lt_34",
        "scope",
        "UnsupportedInterpreterVersionError",
        4_236,
        "expected (scope)",
    ),
    (
        "unconvertable_marker",
        "unconvertable",
        "UnconvertableMarkerError",
        3_453,
        "needs its own pass (not done here)",
    ),
    (
        "invalid_requirement_local_version_label",
        "invalid",
        "InvalidRequirementError",
        262,
        "expected",
    ),
    (
        "metadata_filename_mismatch",
        "invalid",
        "MetadataFilenameMismatchError",
        1_220,
        "needs its own pass (not done here)",
    ),
    (
        "invalid_requirement_wildcard_suffix",
        "invalid",
        "InvalidRequirementError",
        127,
        "expected",
    ),
    (
        "conda_name_too_long",
        "unconvertable",
        "InvalidCondaNameError",
        1_085,
        "expected (unconvertable)",
    ),
    (
        "invalid_version_specifier",
        "invalid",
        "InvalidVersionSpecifierError",
        992,
        "needs its own pass (not done here)",
    ),
    (
        "metadata_unavailable",
        "unavailable",
        "MetadataUnavailable",
        280,
        "harness gap (reroll_data, no sidecar)",
    ),
    (
        "empty_tags_after_abi3_explode",
        "unexpected",
        "IndexError",
        253,
        "BUG (reroll) -- root-caused above",
    ),
    ("invalid_filename", "invalid", "InvalidFilenameError", 239, "expected"),
    ("invalid_abi_tag", "invalid", "InvalidAbiTagError", 205, "expected"),
    (
        "pydantic_validation_leak",
        "unexpected",
        "ValidationError",
        42,
        "BUG (reroll) -- root-caused above",
    ),
    (
        "non_utf8_metadata_blob",
        "unexpected",
        "UnicodeDecodeError",
        31,
        "BUG (reroll_data) -- root-caused above",
    ),
    (
        "markerpry_quote_doubling",
        "unexpected",
        "InvalidRequirement",
        26,
        "BUG (markerpry + reroll) -- root-caused above",
    ),
    ("invalid_name_grammar", "invalid", "InvalidMetadataError", 11, "expected"),
]

summary = pl.DataFrame(
    summary_rows,
    schema=["bucket", "category", "exc_type", "count", "verdict"],
    orient="row",
).with_columns((pl.col("count") / n_error * 100).round(2).alias("share_pct"))

print(
    f"{n_error:,} reroll failures across {n_total:,} total wheel rows ({n_error / n_total:.2%})\n"
)
print(f"{'bucket':<48} {'count':>10}  {'share':>6}  verdict")
print("-" * 110)
for row in summary.sort("count", descending=True).iter_rows(named=True):
    print(
        f"{row['bucket']:<48} {row['count']:>10,}  {row['share_pct']:>5.2f}%  {row['verdict']}"
    )

covered = summary["count"].sum()
print(f"\ncovered {covered:,} / {n_error:,} errors ({covered / n_error:.2%})")

7,649,188 reroll failures across 12,121,854 total wheel rows (63.10%)

bucket                                                count   share  verdict
--------------------------------------------------------------------------------------------------------------
aggregator_mapper_ignores_multi_candidate_parselmouth  4,419,408  57.78%  BUG (reroll)
wheel_version_prerelease_allow_pre_unset          1,463,549  19.13%  config gap (reroll_data)
unsupported_platform_tag                          1,023,613  13.38%  expected (scope)
invalid_interpreter_tag_pypy_etc                    244,466   3.20%  expected (invalid)
dependency_prerelease_allow_pre_unset               178,286   2.33%  config gap (reroll_data)
unsupported_interpreter_major_py2                   142,754   1.87%  expected (scope)
single_mapper_low_confidence                        100,502   1.31%  expected
non_contiguous_python_range                          33,427   0.44%  expected (invalid)
filename_requires_python_mismatch       

## 9. Priority list for a follow-up fix pass

Ordered by impact. Each has a confirmed repro (project/filename in this
corpus, or a self-contained snippet) and a concrete fix location.

| # | Bug | Repo | Impact | Fix location |
|---|---|---|---|---|
| 1 | `aggregator_mapper`'s `if source is PARSELMOUTH: only accept len==1` branch never applies the `probability >= 0.9` threshold the docstring promises for a single mapper's best candidate | `reroll` | **57.8%** of all failures (4,419,408 rows) | `reroll/name_mapping.py:105-112` |
| 2 | `reroll_index_demo`/`reroll_convert` never pass `allow_pre=True` (or thread it through as a CLI flag) | `reroll_data` (harness, not a reroll bug) | 21.5% (1,641,835 rows) | `reroll_data/reroll_index_demo.py::_entry_from_db` |
| 3 | `parse_filename`'s `raise errors[-1]` assumes `errors` is non-empty whenever `configs` is empty; breaks when `tags` is empty before the loop runs (e.g. compound `abi3.abi3t` tag beyond `abi3_upper_bound`) | `reroll` | 253 rows, but crashes instead of categorizing | `reroll/filename/__init__.py:112` |
| 4 | `markerpry`'s `CompareNode.__str__` always wraps `literal` in a fresh, unescaped `"..."`, corrupting round-trips for any literal that itself contains a `"` (reachable via PEP-685 extra-name normalization on legacy metadata) | `markerpry` (+ `reroll`'s unwrapped `Requirement()` re-parses as a second, independent fix) | 26 rows, but a systemic round-trip bug | `markerpry/markerpry/node.py:99,119-120`; `reroll/dependencies/pep508_to_matchspec.py:48` et al. |
| 5 | `parse_metadata`'s `WheelMetadata.model_validate(...)` never catches `pydantic.ValidationError` | `reroll` | 42 rows | `reroll/wheel_metadata.py::parse_metadata` |
| 6 | `_metadata_body` hardcodes `.decode("utf-8")` with no fallback | `reroll_data` | 31 rows | `reroll_data/conda_pypi_index_demo.py:158` |

Buckets marked "expected" in §8 (platform/interpreter scope exclusions,
malformed upstream `Requires-Dist` strings, non-contiguous Python ranges,
etc.) are `reroll` behaving as documented in `docs/errors_and_logging.md`
and are not bugs -- listed there so the *size* of every failure class is
visible, not just the ones worth filing. `UnconvertableMarkerError` (3,453),
`MetadataFilenameMismatchError` (1,220), and `InvalidVersionSpecifierError`
(992) were sampled in §3 but not sub-bucketed further here; §3's cached
samples are the starting point for whoever picks those up next.